# Neural Network Tuning 

Custom torch class

Use cuda for tuning, cpu for training


In [ ]:
import sys
import os
import subprocess

sys.path.append("../")
from src.config import SEED, BASE_PATH, DEVICE
from src.data_utils import get_data
from src.tune_nn import train_and_prelim_eval

print(f"Using device: {DEVICE}")
print(f"Path: {BASE_PATH}")

Import notebook globals

In [ ]:
file_dir = BASE_PATH / "data" / "processed"

DATA_DICT = get_data(is_nomo=False)


LOG_PATH = BASE_PATH / "models" / "logs"
RESULT_PATH = BASE_PATH / "models" / "tune_results"
NUM_PARALLEL_TRIALS = 1

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

## Build + Tune Models

In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(BASE_PATH)
script_path = BASE_PATH / "src" / "tune_nn.py"
n_trials = 2
gpu_idx = 1
cmd = [
    "uv",
    "run",
    str(script_path),
    "--X_path",
    str(BASE_PATH / "data" / "processed" / "base" / "X_train.parquet"),
    "--y_path",
    str(BASE_PATH / "data" / "processed" / "base" / "y_train.xlsx"),
    "--gpu_ids_str",
    str(gpu_idx),
    "--scoring_str",
    "roc_auc",
    "--log_path",
    str(BASE_PATH / "models" / "logs" / "nn.log"),
    "--results_path",
    str(BASE_PATH / "models" / "tune_results" / "nn.json"),
    "--n_trials",
    str(n_trials),
    "--seed",
    str(SEED),
]

## Run tuning

Run once to get things going

In [ ]:
proc = subprocess.Popen(cmd, env=env)
proc.poll()

Run as many times to monitor

In [ ]:
proc.poll()

Run once to terminate

In [ ]:
proc.terminate()

## Train final model

In [ ]:
train_and_prelim_eval(
    data_dict=DATA_DICT,
    json_path=BASE_PATH / "models" / "tune_results" / "nn.json",
    model_save_path=BASE_PATH / "models" / "trained" / "nn.pt",
)